In [2]:
import pandas as pd
import re
import math
import os
import matplotlib.pyplot as plt 
from wordcloud import WordCloud
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import nltk

In [3]:
def clean_text_round(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'\r', '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'[«»"“”]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_keywords_tf_idf(text, top_n=5):
    sentences = sent_tokenize(text)
    
    clean_sentences = [clean_text_round(s) for s in sentences]
    clean_sentences = [s for s in clean_sentences if len(s) > 3]
    
    if not clean_sentences: return []

    total_documents = len(clean_sentences)
    
    stop_words = set(stopwords.words("spanish"))
    stop_words.update(['hotel', 'habitacion', 'alojamiento', 'si', 'mas', 'todo', 'muy', 'nos', 'solo', 'hacer'])
    stemmer = SnowballStemmer("spanish")
    
    frequency_matrix = {}
    stem_to_original = {}

    for sent in clean_sentences:
        freq_table = {}
        words = word_tokenize(sent)
        for word in words:
            word_lower = word.lower()
            if word_lower in stop_words or not word_lower.isalpha():
                continue
            
            root = stemmer.stem(word_lower)
            
            if root not in stem_to_original:
                stem_to_original[root] = word_lower
            
            if root in freq_table:
                freq_table[root] += 1
            else:
                freq_table[root] = 1
        frequency_matrix[sent] = freq_table

    tf_matrix = {}
    for sent, f_table in frequency_matrix.items():
        tf_table = {}
        count_words = len(f_table)
        if count_words == 0: continue
        for word, count in f_table.items():
            tf_table[word] = count / count_words
        tf_matrix[sent] = tf_table

    word_per_doc_table = {}
    for sent, f_table in frequency_matrix.items():
        for word in f_table.keys():
            word_per_doc_table[word] = word_per_doc_table.get(word, 0) + 1
            
    idf_matrix = {}
    for sent, f_table in frequency_matrix.items():
        idf_table = {}
        for word in f_table.keys():
            idf_table[word] = math.log10(total_documents / float(word_per_doc_table[word]))
        idf_matrix[sent] = idf_table

    final_word_scores = {}
    
    for sent, f_table in tf_matrix.items():
        if sent in idf_matrix:
            for word, tf_val in f_table.items():
                idf_val = idf_matrix[sent].get(word, 0)
                score = tf_val * idf_val
                if word in final_word_scores:
                    final_word_scores[word] += score
                else:
                    final_word_scores[word] = score

    sorted_stems = sorted(final_word_scores.items(), key=lambda x: x[1], reverse=True)
    top_keywords = [stem_to_original[stem] for stem, score in sorted_stems[:top_n]]
    
    return top_keywords

df = pd.read_csv("Datos/NLP/reviews_booking.csv")
df['positivo'] = df['positivo'].fillna('')
df['negativo'] = df['negativo'].fillna('')

grouped = df.groupby('hotel').agg({
    'positivo': lambda x: ' '.join(x),
    'negativo': lambda x: ' '.join(x)
}).reset_index()

print(f"{'HOTEL':<25} | {'PALABRAS CLAVE POSITIVAS':<55} | {'PALABRAS CLAVE NEGATIVAS':<40}")
print("-" * 110)

for index, row in grouped.iterrows():
    hotel = row['hotel']
    keywords_pos = get_keywords_tf_idf(row['positivo'], top_n=5)
    keywords_neg = get_keywords_tf_idf(row['negativo'], top_n=5)
    print(f"{hotel:<25} | {', '.join(keywords_pos):<55} | {', '.join(keywords_neg):<40}")

HOTEL                     | PALABRAS CLAVE POSITIVAS                                | PALABRAS CLAVE NEGATIVAS                
--------------------------------------------------------------------------------------------------------------
BilbaoLaVieja             | cama, apartamentos, bien, sofá, hierros                 | cama, sofá, comprobarlo, sabía, recomendamos
BilbaoMuseo               | ubicación, limpio, bien, apartamento, gusto             | demás, cobraron, habitación, funciona, cama
CordobaPatio              | ubicación, limpio, buena, personal, cómodas             | gustó, bien, habitaciones, almohadas, desayuno
Donosti                   | limpio, buena, personal, comodidad, ubicación           | espacio, habitación, bien, baño, desayuno
GranadaCatedral           | ubicacion, gustó, bien, genial, limpio                  | tv, cocinar, bien, pequeño, salón       
MadridPalacioReal         | ubicación, comodidad, limpio, localización, buena       | impersonal, demás, cama, bi

In [4]:
# ==============================================================================
# NUEVO CÓDIGO: GENERACIÓN DE WORDCLOUD GENERAL Y GUARDADO
# ==============================================================================
print("\nGenerando WordCloud general...")

full_text = " ".join(df['positivo']) + " " + " ".join(df['negativo'])

full_text_clean = clean_text_round(full_text)

stop_words_wc = set(stopwords.words("spanish"))
stop_words_wc.update(['hotel', 'habitacion', 'alojamiento', 'si', 'mas', 'todo', 'muy', 'nos', 'solo', 'hacer', "bien", "ser", "día", "vez", "lado"])


wordcloud = WordCloud(
    width=1600, 
    height=800, 
    background_color='white', 
    stopwords=stop_words_wc, 
    colormap='viridis',
    collocations=False
).generate(full_text_clean)

output_path = "Graficos/NLP"
if not os.path.exists(output_path):
    os.makedirs(output_path)
    print(f"Directorio creado: {output_path}")

plt.figure(figsize=(20, 10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')

jpg_path = os.path.join(output_path, "wordcloud_general.jpg")
plt.savefig(jpg_path, format="jpg", bbox_inches='tight')

pdf_path = os.path.join(output_path, "wordcloud_general.pdf")
plt.savefig(pdf_path, format="pdf", bbox_inches='tight')

plt.close()

print(f"WordCloud guardado exitosamente en:\n - {jpg_path}\n - {pdf_path}")


Generando WordCloud general...
WordCloud guardado exitosamente en:
 - Graficos/NLP\wordcloud_general.jpg
 - Graficos/NLP\wordcloud_general.pdf
